In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import ast
from sklearn.metrics import accuracy_score
import copy
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,average_precision_score)
from sklearn.neural_network import MLPClassifier

# Load Data

In [ ]:
newness = **Newness Data**
surprise = **Surprise Data**
value_dict = **Value Data**

In [ ]:
Annotated_cleaned = ** Tabular Data **

# Learning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

In [ ]:
gold_set = **Selected Evaluation Set**

In [ ]:
x_columns=["ID","num_moves"]
y_column="creative_claude"
y=[1 if i=="positive" else 0 for i in gold_set["creative_claude"][2:]]

In [ ]:
Annotated_cleaned_reset = Annotated_cleaned.reset_index()

In [ ]:
merged_df = pd.merge(gold_set, Annotated_cleaned_reset, on='ID', how='left')

In [ ]:
newness_df = np.zeros(merged_df.shape[0]-2, dtype=object)
surprise_df = np.zeros(merged_df.shape[0]-2, dtype=object)
value_df = np.zeros(merged_df.shape[0]-2, dtype=object)
for i in range(merged_df.shape[0]):
    row = merged_df.iloc[i]
    if row["index"]<= 10842:
        continue
    newness_df[i-2] =1 if newness[row["index"]][row["num_moves"]] else 0 
    surprise_df[i-2] =surprise[row["index"]][row["num_moves"]]
    value_df[i-2] = np.clip(value_dict["relative_value_15_cp"][row["index"]][row["num_moves"]], -50, 50)

# Basic

Balanced + Normalized (max min)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error,root_mean_squared_error
from scipy.stats import spearmanr
from sklearn.model_selection import KFold

In [ ]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)

In [ ]:
X = np.stack((newness_df, surprise_normalized,value_normalized), axis=1)
y = np.array(y)

In [ ]:
kf = KFold(n_splits=5,shuffle=True,random_state=24)
kf.get_n_splits()
accs=[]
aucs=[]
precisions=[]
recalls=[]
f1s=[]
for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]
    
    clf = LogisticRegression(max_iter=10000, random_state=0,class_weight="balanced")
    clf.fit(X_train, y_train)

    acc = accuracy_score(y_test,clf.predict(X_test)) 
    auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:,1])
    precision = precision_score(y_test, clf.predict(X_test))
    recall = recall_score(y_test, clf.predict(X_test))
    f1 = f1_score(y_test, clf.predict(X_test))

    accs.append(acc)
    aucs.append(auc)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

print(f"Average acc is {np.mean(accs)}, std is {np.std(accs)}")
print(f"Average auc is {np.mean(aucs)}, std is {np.std(aucs)}")
print(f"Average precision is {np.mean(precisions)}, std is {np.std(precisions)}")
print(f"Average recall is {np.mean(recalls)}, std is {np.std(recalls)}")
print(f"Average f1 is {np.mean(f1s)}, std is {np.std(f1s)}")

# Splines

In [ ]:
surprise_min=np.min(surprise_df)
surprise_max = np.max(surprise_df)
surprise_normalized=(surprise_df - surprise_min)/(surprise_max-surprise_min)
value_min=np.min(value_df)
value_max = np.max(value_df)
value_normalized=(value_df - value_min)/(value_max-value_min)
X = np.stack((newness_df, surprise_normalized,value_normalized), axis=1)

In [ ]:
kf = KFold(n_splits=5,shuffle=True,random_state=24)
kf.get_n_splits()
accs=[]
aucs=[]
precisions=[]
recalls=[]
f1s=[]
for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]
    preprocess = ColumnTransformer([
        ("spline", SplineTransformer(
            degree=3,           # cubic
            n_knots=8,          # tune with CV (e.g., 5–10 common)
            extrapolation="linear",
            include_bias=False  # avoid intercept duplication
        ), [1,2]),
    ], remainder="drop")
    model = make_pipeline(
        preprocess,
        LogisticRegressionCV(cv=5, max_iter=5000, n_jobs=-1,class_weight="balanced",l1_ratios=(0,),use_legacy_attributes=False)
    )

    model.fit(X_train, y_train)
    acc = accuracy_score(y_test,model.predict(X_test)) 
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    precision = precision_score(y_test, model.predict(X_test))
    recall = recall_score(y_test, model.predict(X_test))
    f1 = f1_score(y_test, model.predict(X_test))

    accs.append(acc)
    aucs.append(auc)
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

print(f"Average acc is {np.mean(accs)}, std is {np.std(accs)}")
print(f"Average auc is {np.mean(aucs)}, std is {np.std(aucs)}")
print(f"Average precision is {np.mean(precisions)}, std is {np.std(precisions)}")
print(f"Average recall is {np.mean(recalls)}, std is {np.std(recalls)}")
print(f"Average f1 is {np.mean(f1s)}, std is {np.std(f1s)}")